# P2-01 — GDELT profiling

Profiles `stg_gdelt_raw` (loaded by `warehouse/staging/load_gdelt_raw.py`) to give C-06 the numbers behind each GDELT rule:
reference date, unresolved countries, noise filters, deduplication key, late events.

**Run first** (from the repo root):

```bash
python ingestion/gdelt_events/spike_week.py
python warehouse/staging/load_gdelt_raw.py --reset
```

Set `GDELT_DB` to profile another database (e.g. one month). Outputs written next to this notebook:
`unresolved_country_codes.csv` (for P3).

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]  # notebook lives in warehouse/profiling/
DB = Path(os.environ.get("GDELT_DB", ROOT / "data/warehouse/diplomap.duckdb"))
con = duckdb.connect(str(DB), read_only=True)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_rows", 80)

def q(sql):
    return con.sql(sql).df()

# One view with the derived fields every section uses.
con.execute("""
CREATE OR REPLACE TEMP VIEW ev AS
SELECT *,
       strptime(Day::VARCHAR, '%Y%m%d')::DATE                 AS event_date,
       strptime(DATEADDED::VARCHAR, '%Y%m%d%H%M%S')::DATE     AS added_date
FROM stg_gdelt_raw
""")
TOTAL = con.sql("SELECT count(*) FROM ev").fetchall()[0][0]
q("""SELECT count(*) AS events,
            count(DISTINCT source_file) AS files,
            min(added_date) AS first_added, max(added_date) AS last_added
     FROM ev""")

,events,files,first_added,last_added
0,813679,672,2024-12-17,2024-12-23


## 1. Empty values per column
Columns sorted by share of NULL. Columns at 100 % are never usable; the actor and geo columns drive section 2.

In [2]:
s = q("SUMMARIZE stg_gdelt_raw")[["column_name", "column_type", "null_percentage", "approx_unique"]]
s["null_percentage"] = s["null_percentage"].astype(float)
s.sort_values("null_percentage", ascending=False).reset_index(drop=True)

,column_name,column_type,null_percentage,approx_unique
0,Actor2Type3Code,VARCHAR,99.97,17
1,Actor1Type3Code,VARCHAR,99.94,18
2,Actor2Religion2Code,VARCHAR,99.74,14
3,Actor1Religion2Code,VARCHAR,99.65,15
4,Actor2EthnicCode,VARCHAR,99.54,173
5,Actor1EthnicCode,VARCHAR,99.41,187
6,Actor2KnownGroupCode,VARCHAR,99.14,43
7,Actor1KnownGroupCode,VARCHAR,98.84,46
8,Actor2Religion1Code,VARCHAR,98.72,13
9,Actor1Religion1Code,VARCHAR,98.58,13


## 2. Country of the actors

A country pair needs both `Actor1CountryCode` and `Actor2CountryCode`. The fallback column is the actor's geography
(`ActorNGeo_CountryCode`, FIPS 10-4, to convert with P3's reference).

In [3]:
q("""
SELECT
  round(100 * avg((Actor1CountryCode IS NOT NULL AND Actor2CountryCode IS NOT NULL)::INT), 2) AS pct_both_countries,
  round(100 * avg((Actor1CountryCode IS NULL OR Actor2CountryCode IS NULL)::INT), 2)          AS pct_missing_at_least_one,
  round(100 * avg(((Actor1CountryCode IS NULL AND Actor1Geo_CountryCode IS NOT NULL)
                OR (Actor2CountryCode IS NULL AND Actor2Geo_CountryCode IS NOT NULL))::INT), 2) AS pct_rescuable_by_geo,
  round(100 * avg((coalesce(Actor1CountryCode, 'x') = coalesce(Actor2CountryCode, 'y'))::INT), 2) AS pct_same_country_both_sides,
  round(100 * avg((Actor1CountryCode IS NOT NULL AND Actor2CountryCode IS NOT NULL
                   AND Actor1CountryCode <> Actor2CountryCode)::INT), 2)                     AS pct_interstate_pair
FROM ev
""")

,pct_both_countries,pct_missing_at_least_one,pct_rescuable_by_geo,pct_same_country_both_sides,pct_interstate_pair
0,25.19,74.81,51.07,6.26,18.93


### 2b. Actor types
Classification per actor slot: **country** (country code present), **international org** (type `IGO` or a known group code such as `UNO`, `EUR`, `NAT`),
**non-state / other** (an actor code without country or IGO marker), **none** (no actor).

In [4]:
kind = lambda n: f"""
CASE WHEN Actor{n}CountryCode IS NOT NULL THEN 'country'
     WHEN Actor{n}Type1Code = 'IGO' OR Actor{n}KnownGroupCode IS NOT NULL THEN 'international org'
     WHEN Actor{n}Code IS NOT NULL THEN 'non-state / other'
     ELSE 'none' END"""
q(f"""
SELECT {kind(1)} AS actor1_kind, {kind(2)} AS actor2_kind,
       count(*) AS events, round(100.0 * count(*) / {TOTAL}, 2) AS pct
FROM ev GROUP BY ALL ORDER BY events DESC
""")

,actor1_kind,actor2_kind,events,pct
0,country,country,204965,25.19
1,country,none,128685,15.82
2,country,non-state / other,120101,14.76
3,non-state / other,country,109040,13.40
4,non-state / other,none,104571,12.85
5,non-state / other,non-state / other,60628,7.45
6,none,country,46027,5.66
7,none,non-state / other,31727,3.90
8,international org,country,2753,0.34
9,country,international org,2374,0.29


In [5]:
# Most frequent international organisations and actor type codes among actors without a country.
q("""
SELECT code, count(*) AS events FROM (
  SELECT coalesce(Actor1KnownGroupCode, Actor1Type1Code) AS code FROM ev WHERE Actor1CountryCode IS NULL AND Actor1Code IS NOT NULL
  UNION ALL
  SELECT coalesce(Actor2KnownGroupCode, Actor2Type1Code) FROM ev WHERE Actor2CountryCode IS NULL AND Actor2Code IS NOT NULL)
GROUP BY code ORDER BY events DESC LIMIT 20
""")

,code,events
0,GOV,114876
1,BUS,45277
2,COP,43631
3,CVL,42340
4,EDU,37512
5,NaN,31107
6,JUD,30892
7,LEG,27994
8,MED,24599
9,MIL,21441


### 2c. Country codes not in the CAMEO reference (for P3)
Compared with `DB/CAMEO.country.txt`. Note that this reference also lists regions and cities (`AFR`, `BAG`), which P3 must separate from countries.

In [6]:
con.execute(f"""CREATE OR REPLACE TEMP TABLE cameo_country AS
SELECT * FROM read_csv('{(ROOT / 'DB/CAMEO.country.txt').as_posix()}', delim='\t', header=true)""")
codes = q("""
WITH c AS (SELECT Actor1CountryCode AS gdelt_code FROM ev UNION ALL SELECT Actor2CountryCode FROM ev)
SELECT gdelt_code AS code, count(*) AS actor_mentions, any_value(r.LABEL) AS cameo_label
FROM c LEFT JOIN cameo_country r ON r.CODE = c.gdelt_code
WHERE gdelt_code IS NOT NULL GROUP BY gdelt_code ORDER BY actor_mentions DESC
""")
unresolved = codes[codes.cameo_label.isna()][["code", "actor_mentions"]]
unresolved.to_csv("unresolved_country_codes.csv", index=False)
print(f"{len(codes)} distinct codes, {len(unresolved)} not in CAMEO "
      f"({unresolved.actor_mentions.sum():,} actor mentions) -> unresolved_country_codes.csv")
codes.head(25)

217 distinct codes, 1 not in CAMEO (629 actor mentions) -> unresolved_country_codes.csv


,code,actor_mentions,cameo_label
0,USA,225180,United States
1,GBR,41117,United Kingdom
2,ISR,32787,Israel
3,RUS,27537,Russia
4,CHN,27289,China
5,SYR,23440,Syria
6,CAN,21393,Canada
7,UKR,19126,Ukraine
8,IND,17359,India
9,AUS,16665,Australia


## 3. Event date vs date added
`Day` (SQLDATE) is when the event happened; `DATEADDED` when GDELT first saw it. The gap decides the reference date and how far
past the study period ingestion must go (P2-06).

In [7]:
gap = q("""
SELECT CASE WHEN g < 0 THEN 'negative'
            WHEN g = 0 THEN '0 days' WHEN g = 1 THEN '1 day'
            WHEN g <= 7 THEN '2-7 days' WHEN g <= 30 THEN '8-30 days'
            WHEN g <= 365 THEN '31-365 days' ELSE '> 1 year' END AS gap,
       min(g) AS sort_key, count(*) AS events
FROM (SELECT date_diff('day', event_date, added_date) AS g FROM ev)
GROUP BY 1 ORDER BY sort_key
""").drop(columns="sort_key")
gap["pct"] = 100 * gap.events / TOTAL
gap["cumulative_pct"] = gap.pct.cumsum()
gap

,gap,events,pct,cumulative_pct
0,0 days,795694,97.79,97.79
1,1 day,2322,0.29,98.08
2,2-7 days,5011,0.62,98.69
3,8-30 days,4475,0.55,99.24
4,31-365 days,6001,0.74,99.98
5,> 1 year,176,0.02,100.00


In [8]:
q("""SELECT quantile_cont(date_diff('day', event_date, added_date), [0.5, 0.9, 0.95, 0.99]) AS gap_days_p50_p90_p95_p99 FROM ev""")

,gap_days_p50_p90_p95_p99
0,"[0.0, 0.0, 0.0, 30.0]"


## 4. Root events
`IsRootEvent = 1` marks events from the lead paragraph of the article, a common anti-noise filter.

In [9]:
q(f"""
SELECT IsRootEvent, count(*) AS events, round(100.0 * count(*) / {TOTAL}, 2) AS pct,
       round(avg(NumMentions), 2) AS avg_mentions, round(avg(AvgTone), 2) AS avg_tone
FROM ev GROUP BY 1 ORDER BY 1
""")

,IsRootEvent,events,pct,avg_mentions,avg_tone
0,0,349211,42.92,4.37,-2.44
1,1,464468,57.08,5.13,-1.59


## 5. Mentions, sources and articles
Distribution of the coverage counters; candidate thresholds for the noise filter (P2-04).

In [10]:
dist = q("""
SELECT 'NumMentions' AS measure, quantile_cont(NumMentions, [0.25, 0.5, 0.75, 0.9, 0.99]) AS q, max(NumMentions) AS max FROM ev
UNION ALL SELECT 'NumSources',  quantile_cont(NumSources,  [0.25, 0.5, 0.75, 0.9, 0.99]), max(NumSources)  FROM ev
UNION ALL SELECT 'NumArticles', quantile_cont(NumArticles, [0.25, 0.5, 0.75, 0.9, 0.99]), max(NumArticles) FROM ev
""")
pd.concat([dist.measure, pd.DataFrame(dist.q.tolist(), columns=["p25", "p50", "p75", "p90", "p99"]), dist["max"]], axis=1)

,measure,p25,p50,p75,p90,p99,max
0,NumMentions,2.00,4.00,6.00,10.00,20.00,322
1,NumSources,1.00,1.00,1.00,1.00,2.00,25
2,NumArticles,2.00,4.00,6.00,10.00,16.00,322


In [11]:
# Volume kept by each candidate threshold.
q(f"""
SELECT t AS min_threshold,
       round(100.0 * count(*) FILTER (WHERE NumMentions >= t) / {TOTAL}, 2) AS pct_kept_mentions,
       round(100.0 * count(*) FILTER (WHERE NumSources  >= t) / {TOTAL}, 2) AS pct_kept_sources,
       round(100.0 * count(*) FILTER (WHERE NumSources >= t AND IsRootEvent = 1) / {TOTAL}, 2) AS pct_kept_sources_and_root
FROM ev, range(1, 11) r(t) GROUP BY t ORDER BY t
""")

,min_threshold,pct_kept_mentions,pct_kept_sources,pct_kept_sources_and_root
0,1,100.00,100.00,57.08
1,2,85.97,2.55,1.36
2,3,60.35,0.75,0.39
3,4,53.63,0.39,0.21
4,5,40.52,0.25,0.14
5,6,33.22,0.16,0.10
6,7,24.11,0.13,0.08
7,8,22.82,0.08,0.05
8,9,17.37,0.06,0.04
9,10,17.11,0.05,0.03


## 6. Duplicates
Exact: the same `GlobalEventID` twice. Near: the candidate C-06 key — same day, same actor countries, same CAMEO code.

In [12]:
q(f"""
WITH near AS (
  SELECT count(*) AS n FROM ev
  WHERE Actor1CountryCode IS NOT NULL AND Actor2CountryCode IS NOT NULL
  GROUP BY Day, Actor1CountryCode, Actor2CountryCode, EventCode)
SELECT
  (SELECT count(*) - count(DISTINCT GlobalEventID) FROM ev)  AS exact_duplicate_ids,
  sum(n)::BIGINT AS events_with_country_pair,
  (sum(n) - count(*))::BIGINT AS near_duplicates_removed_by_key,
  round(100.0 * (sum(n) - count(*)) / sum(n), 2) AS pct_of_pair_events,
  (SELECT count(*) - count(DISTINCT (Day, Actor1CountryCode, Actor2CountryCode, EventCode, SOURCEURL))
     FROM ev WHERE Actor1CountryCode IS NOT NULL AND Actor2CountryCode IS NOT NULL) AS near_duplicates_same_url
FROM near
""")

,exact_duplicate_ids,events_with_country_pair,near_duplicates_removed_by_key,pct_of_pair_events,near_duplicates_same_url
0,0,204965,136355,66.53,77664


## 7. Volume per country pair
Undirected pairs of two different countries. Shows how concentrated coverage is (coverage bias, P2-11).

In [13]:
dyads = q("""
SELECT least(Actor1CountryCode, Actor2CountryCode) AS country_a,
       greatest(Actor1CountryCode, Actor2CountryCode) AS country_b,
       count(*) AS events, round(avg(AvgTone), 2) AS avg_tone, round(avg(GoldsteinScale), 2) AS avg_goldstein
FROM ev
WHERE Actor1CountryCode IS NOT NULL AND Actor2CountryCode IS NOT NULL AND Actor1CountryCode <> Actor2CountryCode
GROUP BY 1, 2 ORDER BY events DESC
""")
top = dyads.events.cumsum() / dyads.events.sum()
print(f"{len(dyads):,} pairs; top 10 = {100 * top.iloc[9]:.1f} % of interstate events; "
      f"top 100 = {100 * top.iloc[min(99, len(top) - 1)]:.1f} %; pairs with < 5 events: {(dyads.events < 5).sum():,}")
dyads.head(20)

4,193 pairs; top 10 = 17.7 % of interstate events; top 100 = 49.4 %; pairs with < 5 events: 1,668


,country_a,country_b,events,avg_tone,avg_goldstein
0,RUS,UKR,5851,-5.02,-2.81
1,ISR,PSE,5373,-5.48,-2.59
2,CHN,USA,2727,-2.12,0.31
3,GBR,USA,2678,-0.38,1.59
4,SYR,TUR,2198,-3.35,0.57
5,ISR,SYR,1917,-3.87,-1.60
6,CAN,USA,1878,-1.42,1.31
7,ISR,USA,1700,-3.66,0.14
8,SYR,USA,1488,-3.34,0.40
9,RUS,USA,1403,-3.29,0.00


## 9. Rule options for C-06 (P2-02)

Each candidate rule applied to the events, with its effect on volume and on the KPI inputs (tone, Goldstein, cooperative share).
`st` = **state pairs**: both actor country codes present, neither a CAMEO region or city code, and the two different.
The options are discussed in `gdelt_rules.md`.

In [14]:
REGIONS = ("WSB", "BAG", "GZS", "AFR", "ASA", "BLK", "CRB", "CAU", "CFR", "CAS", "CEU", "EIN", "EAF", "EEU",
           "EUR", "LAM", "MEA", "MDT", "NAF", "NMR", "PGS", "SCN", "SAM", "SAS", "SEA", "SAF", "WAF", "WST")  # first 28 rows of DB/CAMEO.country.txt
con.execute(f"""
CREATE OR REPLACE TEMP VIEW st AS
SELECT *, date_diff('day', event_date, added_date) AS gap
FROM ev
WHERE Actor1CountryCode IS NOT NULL AND Actor2CountryCode IS NOT NULL
  AND Actor1CountryCode NOT IN {REGIONS} AND Actor2CountryCode NOT IN {REGIONS}
  AND Actor1CountryCode <> Actor2CountryCode
""")
kpi = "count(*) AS events, round(avg(AvgTone), 2) AS avg_tone, round(avg(GoldsteinScale), 2) AS avg_goldstein, round(100 * avg((QuadClass IN (1, 2))::INT), 1) AS pct_cooperative"

# R1 — countries: what each step of the country rule removes.
q(f"""
SELECT 'all events' AS step, count(*) AS events FROM ev
UNION ALL SELECT 'both actor country codes present', count(*) FROM ev WHERE Actor1CountryCode IS NOT NULL AND Actor2CountryCode IS NOT NULL
UNION ALL SELECT '... and no region/city code', count(*) FROM ev WHERE Actor1CountryCode IS NOT NULL AND Actor2CountryCode IS NOT NULL AND Actor1CountryCode NOT IN {REGIONS} AND Actor2CountryCode NOT IN {REGIONS}
UNION ALL SELECT '... and two different countries (state pairs)', count(*) FROM st
UNION ALL SELECT 'upper bound added by actor-geo fallback', count(*) FROM ev
  WHERE (Actor1CountryCode IS NULL AND Actor1Code IS NOT NULL AND Actor1Geo_CountryCode IS NOT NULL AND Actor2CountryCode IS NOT NULL AND Actor2CountryCode NOT IN {REGIONS})
     OR (Actor2CountryCode IS NULL AND Actor2Code IS NOT NULL AND Actor2Geo_CountryCode IS NOT NULL AND Actor1CountryCode IS NOT NULL AND Actor1CountryCode NOT IN {REGIONS})
""")

,step,events
0,all events,813679
1,both actor country codes present,204965
2,... and no region/city code,190819
3,... and two different countries (state pairs),140292
4,upper bound added by actor-geo fallback,224360


In [15]:
# R2 — reference date and late events, on state pairs: share of events reported within k days of the event.
q("""
SELECT k AS max_days_late, round(100 * avg((gap <= k)::INT), 2) AS pct_state_pair_events
FROM st, (VALUES (0), (1), (7), (15), (30), (365)) v(k) GROUP BY k ORDER BY k
""")

,max_days_late,pct_state_pair_events
0,0,97.33
1,1,97.66
2,7,98.39
3,15,98.39
4,30,99.25
5,365,99.99


In [16]:
# R3 — noise filters on state pairs.
q(f"""
SELECT 'none' AS filter, {kpi} FROM st
UNION ALL SELECT 'IsRootEvent = 1', {kpi} FROM st WHERE IsRootEvent = 1
UNION ALL SELECT 'NumMentions >= 2', {kpi} FROM st WHERE NumMentions >= 2
UNION ALL SELECT 'NumMentions >= 3', {kpi} FROM st WHERE NumMentions >= 3
UNION ALL SELECT 'IsRootEvent = 1 and NumMentions >= 2', {kpi} FROM st WHERE IsRootEvent = 1 AND NumMentions >= 2
UNION ALL SELECT 'NumSources >= 2', {kpi} FROM st WHERE NumSources >= 2
""")

,filter,events,avg_tone,avg_goldstein,pct_cooperative
0,none,140292,-1.94,0.81,74.20
1,IsRootEvent = 1,75157,-1.46,0.91,75.40
2,NumMentions >= 2,107493,-1.87,0.90,74.70
3,NumMentions >= 3,58706,-1.79,0.86,74.20
4,IsRootEvent = 1 and NumMentions >= 2,60414,-1.42,0.96,75.50
5,NumSources >= 2,3579,-2.93,0.31,68.80


In [17]:
# R4 — deduplication keys on state pairs: events kept by each key.
keys = {
    "GlobalEventID (exact)": "GlobalEventID",
    "day + directed pair + EventCode + SOURCEURL": "Day, Actor1CountryCode, Actor2CountryCode, EventCode, SOURCEURL",
    "day + undirected pair + EventRootCode + SOURCEURL": "Day, least(Actor1CountryCode, Actor2CountryCode), greatest(Actor1CountryCode, Actor2CountryCode), EventRootCode, SOURCEURL",
    "day + directed pair + EventCode (no URL)": "Day, Actor1CountryCode, Actor2CountryCode, EventCode",
}
pd.DataFrame([
    (name, *con.sql(f"SELECT count(DISTINCT ({k})), round(100.0 * count(DISTINCT ({k})) / count(*), 1), "
                    f"(SELECT round(avg(t), 2) FROM (SELECT any_value(AvgTone) t FROM st GROUP BY {k})) FROM st").fetchall()[0])
    for name, k in keys.items()], columns=["key", "events_kept", "pct_kept", "avg_tone_after"])

,key,events_kept,pct_kept,avg_tone_after
0,GlobalEventID (exact),140292,100.00,-1.94
1,day + directed pair + EventCode + SOURCEURL,81466,58.10,-1.95
2,day + undirected pair + EventRootCode + SOURCEURL,61540,43.90,-2.25
3,day + directed pair + EventCode (no URL),50676,36.10,-1.52


In [18]:
# Funnel of the accepted rules, as warehouse/staging/gdelt.sql applies them.
q("""
WITH p AS (SELECT * FROM st WHERE event_date BETWEEN DATE '2024-12-17' AND DATE '2025-07-15'),
     f AS (SELECT * FROM p WHERE gap <= 30),
     n AS (SELECT * FROM f WHERE NumMentions >= 2),
     d AS (SELECT * FROM n QUALIFY row_number() OVER (
             PARTITION BY Day, Actor1CountryCode, Actor2CountryCode, EventCode, SOURCEURL
             ORDER BY DATEADDED, GlobalEventID) = 1)
SELECT 'R1 state pairs' AS step, count(*) AS events FROM st
UNION ALL SELECT 'R2 dated inside the study period', count(*) FROM p
UNION ALL SELECT 'R2 reported within 30 days', count(*) FROM f
UNION ALL SELECT 'R3 NumMentions >= 2', count(*) FROM n
UNION ALL SELECT 'R4 dedup (day, pair, code, URL)', count(*) FROM d
""")

,step,events
0,R1 state pairs,140292
1,R2 dated inside the study period,136932
2,R2 reported within 30 days,136932
3,R3 NumMentions >= 2,104975
4,"R4 dedup (day, pair, code, URL)",68987


## 8. Key numbers for table_contracts/README.md
One place to copy from once the profiling is reviewed.

In [19]:
one = lambda sql: con.sql(sql).fetchall()[0][0]
pd.Series({
    "events": TOTAL,
    "% interstate pair (both countries, different)": one("SELECT 100 * avg((Actor1CountryCode IS NOT NULL AND Actor2CountryCode IS NOT NULL AND Actor1CountryCode <> Actor2CountryCode)::INT) FROM ev"),
    "% missing at least one actor country": one("SELECT 100 * avg((Actor1CountryCode IS NULL OR Actor2CountryCode IS NULL)::INT) FROM ev"),
    "% root events": one("SELECT 100 * avg(IsRootEvent) FROM ev"),
    "% event date > 7 days before added": one("SELECT 100 * avg((date_diff('day', event_date, added_date) > 7)::INT) FROM ev"),
    "median NumMentions": one("SELECT median(NumMentions) FROM ev"),
    "exact duplicate ids": one("SELECT count(*) - count(DISTINCT GlobalEventID) FROM ev"),
    "unresolved country codes (vs CAMEO)": len(unresolved),
}).to_frame("value")

,value
events,"813,679.00"
"% interstate pair (both countries, different)",18.93
% missing at least one actor country,74.81
% root events,57.08
% event date > 7 days before added,1.31
median NumMentions,4.00
exact duplicate ids,0.00
unresolved country codes (vs CAMEO),1.00
